In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "Token"
token = UserSecretsClient().get_secret(secret_label)

! git clone https://{token}@github.com/PietroGhersetich/RecSys_challenge_2025

Cloning into 'RecSys_challenge_2025'...
remote: Enumerating objects: 1043, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 1043 (delta 51), reused 0 (delta 0), pack-reused 961 (from 2)
Receiving objects: 100% (1043/1043), 408.97 MiB | 48.77 MiB/s, done.
Resolving deltas: 100% (571/571), done.


In [2]:
! pip install PyGithub requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.4 MB/s eta 0:00:00


In [3]:
! cd /kaggle/working/RecSys_challenge_2025 && python run_compile_all_cython.py

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorization_Cython_Epoch.pyx... 
In file included from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarraytypes.h:1929,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/arrayobject.h:5,
                 from MatrixFactorization_Cython_Epoch.c:1252:
/usr/local/lib/python3.11/dist-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
   17 | #warning "Using deprecated NumPy API, disable it with " \
      |  ^~~~~~~
Matrix

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import random
import tqdm
import gc
import time
import os
import shutil
import optuna
from optuna.distributions import FloatDistribution, IntDistribution, CategoricalDistribution
import scipy.sparse as sps

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/sample_submission.csv
/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv


In [5]:
from github import Github
from github import Auth

auth = Auth.Token(token)
g = Github(auth=auth)

repo = None
for r in g.get_user().get_repos():
    if r.name == 'RecSys_challenge_2025':
        repo = r
        print('Repository found')

Repository found


In [6]:
def upload_file(filepath_kaggle, filepath_github, commit_message):
    try:
        contents = repo.get_contents(filepath_github)
        with open(filepath_kaggle, "rb") as file:
            repo.update_file(contents.path, commit_message, file.read(), contents.sha)
    except Exception:
        with open(filepath_kaggle, "rb") as file:
            repo.create_file(filepath_github, commit_message, file.read())

In [7]:
db_name = 'knn_study.db'
json_name_item = 'item_knn_best_params.json'
json_name_user = 'user_knn_best_params.json'

config = {
    'tune_parameters_item_knn': True,
    'tune_parameters_user_knn': True,
    'database_name': db_name,
    'database_path': f'/kaggle/working/{db_name}',
    'save_json': True,
    'json_name_item': json_name_item,
    'json_name_user': json_name_user,
    'copy_prev_tuning_db': True,
    'train_best_model': False,
    'copy_prev_best_params': True,
    'save_github': True,
    'candidates': '50',
    'n_trials_item': 100, # 1000 -> 5h
    'n_trials_user': 100  # 300 -> 5h
}

In [8]:
try:
    shutil.copyfile(
        f'/kaggle/working/RecSys_challenge_2025/tuning_results/recall@{config["candidates"]}/{config["database_name"]}', 
        config['database_path']
    )
except FileNotFoundError:
    pass # if not present optuna will create it

# Data Import

In [9]:
train_data = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_train.csv')
user_ids_test = pd.read_csv('/kaggle/input/recommender-systems-2025-challenge-polimi/data_target_users_test.csv')

In [10]:
data = np.ones(len(train_data))

URM_all = sps.csr_matrix((data, (train_data['row'].values, train_data['col'].values)),
                         shape=(train_data['row'].max() + 1, train_data['col'].max() + 1))

URM_all

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3043058 stored elements and shape (27095, 6969)>

In [11]:
%cd /kaggle/working/RecSys_challenge_2025

/kaggle/working/RecSys_challenge_2025


In [12]:
from Recommenders.BaseRecommender import BaseRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

from Evaluation.Evaluator import EvaluatorHoldout

np.int = int
np.random.seed(42)
np.bool = bool
np.float = float

In [13]:
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_all, train_percentage = 0.8)

In [14]:
import sqlite3

def print_all_rows(table_name, database_path):
    connection = sqlite3.connect(database_path)
    cursor = connection.cursor()

    try:
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        print(f"All rows from table '{table_name}':")
        for row in rows:
            print(row)

    except sqlite3.Error as e:
        print(f"Error reading data from table '{table_name}': {e}")

    finally:
        cursor.close()
        connection.close()

print_all_rows('studies', config['database_path'])

All rows from table 'studies':
(1, 'item_knn_Optimization')
(2, 'hyperparameters_tuning_knn_item')
(3, 'user_knn_Optimization')
(4, 'item_knn_fine_tuning_Optimization')
(5, 'user_knn_fine_tuning_Optimization')


In [15]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_itemknn(optuna_trial):
    recommender_instance = ItemKNNCFRecommender(URM_train)
    
    params = {
        'similarity': 'asymmetric',
        'topK': optuna_trial.suggest_int("topK", 5, 50), # Old: [10, 1000]
        'shrink': optuna_trial.suggest_int("shrink", 300, 600), # Old: [0, 500]
        'normalize': True,
        'feature_weighting': 'TF-IDF',
        'asymmetric_alpha': optuna_trial.suggest_float("asymmetric_alpha", 0.15, 0.35), # Old: [0, 2.0]
    }
    
    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [16]:
%%time
# an Itemknn study_name MUST contain "item_knn"
if config['tune_parameters_item_knn']:
    study = optuna.create_study(direction='maximize', study_name='item_knn_fine_tuning_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_itemknn, n_trials=config['n_trials_item'])

[I 2025-12-25 11:27:40,300] Using an existing study with name 'item_knn_fine_tuning_Optimization' instead of creating a new one.


Previous trials found: 200
Similarity column 6969 (100.0%), 1991.18 column/sec. Elapsed time 3.50 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.42 sec. Users per second: 1648


[I 2025-12-25 11:28:00,775] Trial 200 finished with value: 0.3974330776808815 and parameters: {'topK': 20, 'shrink': 336, 'asymmetric_alpha': 0.19445307115916233}. Best is trial 171 with value: 0.39883415846571035.


Similarity column 6969 (100.0%), 1975.32 column/sec. Elapsed time 3.53 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 15.94 sec. Users per second: 1698


[I 2025-12-25 11:28:20,635] Trial 201 finished with value: 0.3984878190630991 and parameters: {'topK': 17, 'shrink': 340, 'asymmetric_alpha': 0.20277836235098678}. Best is trial 171 with value: 0.39883415846571035.


...

Similarity column 6969 (100.0%), 2026.97 column/sec. Elapsed time 3.44 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 15.88 sec. Users per second: 1704


[I 2025-12-25 12:00:20,036] Trial 298 finished with value: 0.39822397544227783 and parameters: {'topK': 15, 'shrink': 352, 'asymmetric_alpha': 0.2111267869154416}. Best is trial 171 with value: 0.39883415846571035.


Similarity column 6969 (100.0%), 2000.42 column/sec. Elapsed time 3.48 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 16.17 sec. Users per second: 1674


[I 2025-12-25 12:00:40,089] Trial 299 finished with value: 0.3977683660984046 and parameters: {'topK': 16, 'shrink': 384, 'asymmetric_alpha': 0.22569474310497636}. Best is trial 171 with value: 0.39883415846571035.


CPU times: user 31min 49s, sys: 1min 14s, total: 33min 4s
Wall time: 33min 1s


In [17]:
if config['save_json'] and config['save_github'] and config['tune_parameters_item_knn']:
    j_name = config['json_name_item']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "item_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

No update needed. Best: 0.398834 <= Saved: 0.398834


In [18]:
evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[int(config['candidates'])])

def objective_function_userknn(optuna_trial):
    recommender_instance = UserKNNCFRecommender(URM_train)
    
    # 1. Choose the similarity metric first
    similarity = 'asymmetric' # Old: ['cosine', 'asymmetric', 'dice', 'jaccard', 'tanimoto', 'tversky', 'euclidean']
    
    # 2. Global parameters
    params = {
        'similarity': similarity,
        'topK': optuna_trial.suggest_int("topK", 280, 520), # Old: [10, 1000]
        'shrink': optuna_trial.suggest_int("shrink", 170, 315), # Old: [0, 500]
        'normalize': True, # Old: [True, False]
        'feature_weighting': "TF-IDF", # Old: ["BM25", "TF-IDF", "none"]
    }
    
    # 3. Conditional parameters based on similarity
    # Since similarity is fixed to 'asymmetric', we directly set its specific parameter
    params["asymmetric_alpha"] = optuna_trial.suggest_float("asymmetric_alpha", 0.12, 0.26, log=False) # Old: [0, 2.0]

    recommender_instance.fit(**params)
    
    result_df, _ = evaluator.evaluateRecommender(recommender_instance)
    
    return result_df.loc[int(config['candidates'])]["RECALL"]

EvaluatorHoldout: Ignoring 28 ( 0.1%) Users that have less than 1 test interactions


In [19]:
%%time
# an Userknn study_name MUST contain "user_knn"
if config['tune_parameters_user_knn']:
    study = optuna.create_study(direction='maximize', study_name='user_knn_fine_tuning_Optimization', 
                                storage=f'sqlite:///{config["database_path"]}', load_if_exists=True)
    print(f"Previous trials found: {len(study.trials)}")
    study.optimize(objective_function_userknn, n_trials=config['n_trials_user'])

[I 2025-12-25 12:00:42,404] Using an existing study with name 'user_knn_fine_tuning_Optimization' instead of creating a new one.


Previous trials found: 200
Similarity column 27095 (100.0%), 736.75 column/sec. Elapsed time 36.78 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 23.80 sec. Users per second: 1137


[I 2025-12-25 12:01:43,834] Trial 200 finished with value: 0.36676667154503656 and parameters: {'topK': 397, 'shrink': 192, 'asymmetric_alpha': 0.21027019565720254}. Best is trial 172 with value: 0.3671444777085252.


Similarity column 27095 (100.0%), 730.64 column/sec. Elapsed time 37.08 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 23.24 sec. Users per second: 1165


[I 2025-12-25 12:02:44,920] Trial 201 finished with value: 0.3669861303751008 and parameters: {'topK': 395, 'shrink': 188, 'asymmetric_alpha': 0.19618003464489986}. Best is trial 172 with value: 0.3671444777085252.


...

Similarity column 27095 (100.0%), 748.62 column/sec. Elapsed time 36.19 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 22.35 sec. Users per second: 1211


[I 2025-12-25 13:39:44,833] Trial 298 finished with value: 0.36709681195606464 and parameters: {'topK': 400, 'shrink': 207, 'asymmetric_alpha': 0.18571905500365268}. Best is trial 277 with value: 0.36716240832167824.


Similarity column 27095 (100.0%), 751.19 column/sec. Elapsed time 36.07 sec
EvaluatorHoldout: Processed 27067 (100.0%) in 22.49 sec. Users per second: 1204


[I 2025-12-25 13:40:44,126] Trial 299 finished with value: 0.36681856899985266 and parameters: {'topK': 403, 'shrink': 207, 'asymmetric_alpha': 0.18628571583128375}. Best is trial 277 with value: 0.36716240832167824.


CPU times: user 1h 38min 57s, sys: 1min 7s, total: 1h 40min 5s
Wall time: 1h 40min 1s


In [20]:
if config['save_json'] and config['save_github'] and config['tune_parameters_user_knn']:
    j_name = config['json_name_user']
    l_path = f"/kaggle/working/{j_name}"
    r_path = f"tuning_results/recall@{config['candidates']}/best_params/{j_name}"
    
    best_val, best_params, best_trial_num, best_study_name = -float('inf'), {}, None, None
    
    try:
        for s in optuna.get_all_study_summaries(storage=f"sqlite:///{config['database_path']}"):
            if "user_knn" not in s.study_name: 
                continue

            if s.best_trial and s.best_trial.value > best_val:
                best_val, best_params, best_trial_num, best_study_name = s.best_trial.value, s.best_trial.params, s.best_trial.number, s.study_name
    except:
        best_val, best_params, best_trial_num, best_study_name = study.best_value, study.best_params, study.best_trial.number, study.study_name

    prev_val, exists = -float('inf'), False
    try:
        prev_val = json.loads(repo.get_contents(r_path).decoded_content.decode()).get('value', -float('inf'))
        exists = True
    except:
        pass

    if not exists or best_val > prev_val:
        print(f"Updating JSON. Best: {best_val:.6f} (Prev: {prev_val:.6f})")
        with open(l_path, 'w') as f:
            json.dump({"value": best_val, "study_name": best_study_name, "trial_number": best_trial_num, "params": best_params}, f, indent=4)
        if config.get('save_github'):
            upload_file(l_path, r_path, f'Update params {j_name} (from kaggle notebook)')
    else:
        print(f"No update needed. Best: {best_val:.6f} <= Saved: {prev_val:.6f}")

Updating JSON. Best: 0.367162 (Prev: 0.367144)


In [21]:
if config['save_github'] and (config['tune_parameters_user_knn'] or config['tune_parameters_item_knn']):
    upload_file(
        config['database_path'], 
        f'tuning_results/recall@{config["candidates"]}/{config["database_name"]}',
        'Tuning db updated results (from kaggle notebook)'
    )